In [ ]:
"""
Tiny GPT From Scratch

Training a ~35M parameter decoder-only Transformer on the TinyStories dataset
using PyTorch.

Features:
- Byte Pair Encoding tokenizer
- Streaming dataset pipeline
- Mixed precision training
- Cosine learning rate decay
- Checkpoint saving and resume support
"""

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import time
from torch.amp import autocast, GradScaler

ckpt_dir = "/content/drive/MyDrive/gpt_checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

block_size=256
batch_size=32
learning_rate=3e-4
max_iters = 21002
warmup_iters = 2000
base_lr = learning_rate


eval_iters=100
eval_interval=200
n_embd = 512
accum_steps = 4
n_head=8
n_layer=8
dropout=0.1

cuda


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True
)



In [ ]:
def infinite_batch_stream(dataset, block_size, batch_size):
    while True:
        yield from stream_batches(dataset, block_size, batch_size)


In [ ]:
sample = next(__builtins__.iter(dataset))
print(sample["text"][:200])


One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = ByteLevel()

trainer = BpeTrainer(
    vocab_size=16000,
    special_tokens=["<unk>"]
)

# collect some text samples
texts = []
for i, sample in enumerate(dataset):
    texts.append(sample["text"])
    if i >= 2000:
        break

tokenizer.train_from_iterator(texts, trainer)


In [ ]:

dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True
)


In [ ]:
vocab_size = tokenizer.get_vocab_size()
print("BPE vocab size:", vocab_size)

encode = lambda s: tokenizer.encode(s).ids
decode = lambda l: tokenizer.decode(l)


BPE vocab size: 9851


In [ ]:
def stream_batches(dataset, block_size, batch_size):
    buffer = []
    batch_x = []
    batch_y = []

    for sample in dataset:
        buffer.extend(encode(sample["text"]))

        while len(buffer) >= block_size + 1:
            x = buffer[:block_size]
            y = buffer[1:block_size + 1]
            buffer = buffer[block_size:]

            batch_x.append(x)
            batch_y.append(y)

            if len(batch_x) == batch_size:
                yield (
                    torch.tensor(batch_x, dtype=torch.long),
                    torch.tensor(batch_y, dtype=torch.long)
                )
                batch_x = []
                batch_y = []

In [ ]:
train_dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True
).shard(num_shards=20, index=0)

val_dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True
).shard(num_shards=20, index=1)

train_stream = infinite_batch_stream(train_dataset, block_size, batch_size)
val_stream   = infinite_batch_stream(val_dataset, block_size, batch_size)



val_batches = 1000
val_data = [next(val_stream) for _ in range(val_batches)]

def get_batch(split):
    if split == "train":
        return next(train_stream)
    else:
        return val_data[torch.randint(len(val_data), (1,)).item()]


In [ ]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size, device=device))) # Fix applied here

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # (B, T, F) -> (B, T, [h1, h1, h1, h1, h2, h2, h2, h2, h3, h3, h3, h3])
        out = self.dropout(self.proj(out))
        return out


class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)


        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape


        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            logits, loss = self.forward(index_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            logits = logits / 1.0  # temperature
            v, ix = torch.topk(logits, 50)
            probs = F.softmax(v, dim=-1)
            index_next = torch.gather(ix, -1, torch.multinomial(probs, 1)) # Fixed here

            index = torch.cat((index, index_next), dim=1)
        return index

model = GPTLanguageModel(vocab_size)

m = model.to(device)

In [ ]:
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      X, Y = X.to(device), Y.to(device) 
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
use_amp = device.type == "cuda"
scaler = GradScaler(enabled=use_amp)

# Resume from checkpoint
import glob
import os

start_step = 0
ckpt_dir = "/content/gdrive/MyDrive/gpt_checkpoints/"

ckpt_pattern = os.path.join(ckpt_dir, "checkpoint_step_*.pt")


checkpoints = glob.glob(ckpt_pattern)



In [ ]:
print("Checkpoint dir exists:", os.path.exists(ckpt_dir))
print("Checkpoints found:", glob.glob(ckpt_pattern))


Checkpoint dir exists: True
Checkpoints found: ['/content/gdrive/MyDrive/gpt_checkpoints/checkpoint_step_9000.pt', '/content/gdrive/MyDrive/gpt_checkpoints/checkpoint_step_15000.pt', '/content/gdrive/MyDrive/gpt_checkpoints/checkpoint_step_20000.pt', '/content/gdrive/MyDrive/gpt_checkpoints/checkpoint_step_21000.pt']


In [ ]:
if checkpoints:
    latest_ckpt = max(
        checkpoints,
        key=lambda x: int(x.split("_")[-1].split(".")[0])
    )

    checkpoint = torch.load(latest_ckpt, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    start_step = checkpoint["step"]

    print(f"✓ Resumed from {latest_ckpt}")
else:
    print("No checkpoint found. Starting from scratch.")



if start_step == 0:
    print("\nRunning speed benchmark...")
    model.train()
    start = time.time()

    for step in range(100):
        xb, yb = get_batch('train')
        xb, yb = xb.to(device), yb.to(device)

        with autocast(device_type="cuda", enabled=use_amp):
            logits, loss = model(xb, yb)
            loss = loss / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

    elapsed = time.time() - start
    time_per_step = elapsed / 100

    print(f"Time per step: {time_per_step:.3f}s")
    print(f"Estimated time for 1K steps: {time_per_step * 1000 / 60:.1f} minutes")
    print(f"Estimated time for 10K steps: {time_per_step * 10000 / 3600:.1f} hours")
    print("Benchmark complete. Starting main training...\n")

    # Reset optimizer
    optimizer.zero_grad(set_to_none=True)

✓ Resumed from /content/gdrive/MyDrive/gpt_checkpoints/checkpoint_step_21000.pt


In [ ]:
import math

def get_lr(step):
    max_lr = base_lr      # peak learning rate (3e-4)
    min_lr = 3e-5         # final learning rate after decay
    decay_start = 15000   # step where cosine decay begins
    decay_end = 20000     # step where decay ends

    # linear warmup
    if step < warmup_iters:
        return max_lr * step / warmup_iters

 
    if step < decay_start:
        return max_lr

    # cosine decay from max_lr down to min_lr
    if step <= decay_end:
        decay_ratio = (step - decay_start) / (decay_end - decay_start)
        cosine_decay = 0.5 * (1 + math.cos(math.pi * decay_ratio))
        return min_lr + (max_lr - min_lr) * cosine_decay


    return min_lr



for step in range(start_step, max_iters):

    opt_step = step // accum_steps
    lr = get_lr(opt_step)

    for param_group in optimizer.param_groups:
        param_group['lr'] = lr


    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)

    with autocast(device_type="cuda",enabled=use_amp):
        logits, loss = model(xb, yb)
        loss = loss / accum_steps

    scaler.scale(loss).backward()

    if (step + 1) % accum_steps == 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    # Checkpoint saving outside gradient accumulation block
    if step % 1000 == 0 and step > 0:
        checkpoint = {
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
        }
        torch.save(checkpoint, f"{ckpt_dir}/checkpoint_step_{step}.pt")
        print(f"✓ Saved checkpoint at step {step}")

    if step % eval_interval == 0:
        losses = estimate_loss()
        print(
            f"step: {step}, "
            f"train loss: {losses['train']:.3f}, "
            f"val loss: {losses['val']:.3f}"
        )

print(f"\nFinal loss: {loss.item():.3f}")
print(f"Peak memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

✓ Saved checkpoint at step 21000
step: 21000, train loss: 1.724, val loss: 1.789

Final loss: 0.546
Peak memory: 5.44 GB


In [ ]:
model.eval()
with torch.no_grad():
    xb, yb = get_batch("val")   # or "train"
    xb, yb = xb.to(device), yb.to(device)
    logits, loss = model(xb, yb)

print("Loss:", loss.item())


Loss: 2.0991387367248535


In [ ]:
prompt = "Once upon a time"
context = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
out = model.generate(context, max_new_tokens=100)[0].tolist()
text = decode(out)
print(text.replace("Ġ", " ").replace("Ċ", "\n"))

 Once  upon  a  time ,  there  was  a  little  girl  named  Lily .  She  had  a  toy  cat  named  Mittens .  Mittens  had  a  soft  fur .  One  day ,  Lily 's  friend ,  Mittens ,  came  to  play .  Mittens  saw  the  mouse  and  said ,  " I  like  your  cat  look  pretty .  Can  I  pet  it ?"  The  mouse  looked  at  Lily  and  said ,  " Sure !  It  is  so  soft  and  warm ." 
 
 Lily  and  Mittens  played  together  all  day  long .  Mittens  would  rub  Lily 's  fur  and  it  would  all  look  at  herself .  Lily


In [ ]:
model.eval()
with torch.no_grad():
    out = model.generate(
        context,
        max_new_tokens=150,
    )[0].tolist()

text = decode(out)
print(text.replace("Ġ", " ").replace("Ċ", "\n"))


 Once  upon  a  time  there  was  a  little  boy  called  Tom .  He  loved  to  play  outside  all  day  long .  But  one  day  Tom  went  to  the  park  to  play .  In  the  park  he  met  a  big  truck .  The  truck  was  very  big  and  strong .  The  truck  driver  said  hello  to  Tom  and  invited  him  to  come  look .  
 
 Tom  did  not  know  what  the  truck  was ,  but  he  was  curious .  He  went  to  meet  the  truck  driver .  The  truck  driver  told  Tom  all  about  his  adventure .  Tom  was  very  confused .  
 
 The  truck  driver  said  he  was  looking  for  his  friend 's  birthday  party .  Tom  felt  embarrassed  but  tried  to  help  the  truck  driver .  When  the  truck  was  gone ,  the  truck  driver  said  he  had  already  received  a  new  house .  The  truck  driver  said  he  couldn 't


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


Total parameters: 35,436,155
